# dataclass-training-args — worked example 1: WandbArgs dataclass with run-name derivation and dataclasses.replace clone

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dataclass-training-args`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A `@dataclass` bundles training hyperparameters so the trainer signature stays flat. `__post_init__` runs right after the auto-generated `__init__`, which is where you derive computed fields and validate. `dataclasses.replace(obj, **changes)` builds a *new* instance with some fields overridden, re-running `__post_init__` on the copy.

## Worked solution

**Goal:** define a `WandbArgs` dataclass that validates its fields, auto-derives a `run_name` when none is given, and supports cloning with overrides.

**Step 1 — declare fields with defaults.** We give `project`, `entity`, `lr`, and `seed` defaults plus a `run_name: str | None = None`. Declaring `run_name` as optional lets the caller leave it blank so we can synthesize one.

**Step 2 — validate in `__post_init__`.** This method runs after `__init__` assigns every field, so all values are present. We raise `ValueError` for a non-positive `lr` and a negative `seed`. Doing it here (not in the trainer) keeps the failure close to the bad config.

**Step 3 — derive `run_name` if absent.** Still inside `__post_init__`, if `run_name is None` we build one from the project and seed (e.g. `demo-seed7`). Because `__post_init__` can assign to `self`, this computed default is available on every instance.

**Step 4 — clone with `replace`.** `dataclasses.replace(args, seed=99)` returns a brand new `WandbArgs` with `seed` overridden. Crucially it re-runs `__post_init__`, so the clone gets its own freshly-derived `run_name` (`demo-seed99`), not the original's. This is the idiomatic way to sweep one hyperparameter without mutating the base config.

**Why it works:** the dataclass machinery generates `__init__` from the field declarations, then calls `__post_init__` once; `replace` simply calls the constructor again with merged kwargs, so all the derivation and validation logic fires automatically on the copy.

In [ ]:
from dataclasses import dataclass, replace

@dataclass
class WandbArgs:
    project: str = 'demo'
    entity: str = 'team'
    lr: float = 3e-4
    seed: int = 0
    run_name: str | None = None

    def __post_init__(self):
        if self.lr <= 0:
            raise ValueError(f'lr must be > 0, got {self.lr}')
        if self.seed < 0:
            raise ValueError(f'seed must be >= 0, got {self.seed}')
        if self.run_name is None:
            self.run_name = f'{self.project}-seed{self.seed}'

base = WandbArgs(project='demo', seed=7)
clone = replace(base, seed=99)
print(base.run_name)
print(clone.run_name)
print(base.run_name != clone.run_name)